In [28]:

import os
#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.98"

import jax
import jax.numpy as jnp

import numpy as np
import matplotlib.pyplot as plt
from dmri.simulators import  Ball3StickSharedDiffusivity
from dmri.simulators.acquisition_scheme import acquisition_scheme
from flax import nnx

import matplotlib.pyplot as plt

plt.style.use("../../dmri/utils/pyloric.mplstyle")


In [29]:
pits_short_mask = np.load('outputs/coverage_eval2/pit_coverage_mask_short.npz', allow_pickle=True)
pits_short_theta = np.load('outputs/coverage_eval2/pit_coverage_theta_short.npz', allow_pickle=True)
pits_long_mask = np.load('outputs/coverage_eval2/pit_coverage_mask_long.npz', allow_pickle=True)
pits_long_theta = np.load('outputs/coverage_eval2/pit_coverage_theta_long.npz', allow_pickle=True)
# Access individual arrays
alpha_s = pits_short_mask['alpha']
coverage_s = pits_short_mask['coverage']
coverage_l = pits_long_mask['coverage']
pits_s = pits_short_mask['pits']
pits_l = pits_long_mask['pits']
labels_s = pits_short_mask['labels']
hyperparameters_s = pits_short_mask['hyperparameters']

In [30]:
pits_l.shape

(11, 20000)

In [31]:
alpha = pits_short_theta['alpha']
thetas = pits_short_theta['coverage']
pits = pits_short_theta['pits']
pits_theta_long = pits_long_theta['pits']
labels = pits_short_theta['labels']
theta_mask_counts = pits_short_theta['theta_mask_counts']
theta_mask_coverage = pits_short_theta['theta_mask_coverage']
theta_mask_errors = pits_short_theta['theta_mask_errors']
theta_mask_labels = pits_short_theta['theta_mask_labels']
theta_masks = pits_short_theta['theta_mask_support']

In [32]:
from coverage_eval_conv import mask_to_label

In [33]:
from dmri.train.utils import load_cfg, load_checkpoint
checkpoint, model, simulator = load_checkpoint("/home/macke/mgloeckler90/dmri/results/updated_all_3_6_8_128_model_prior_with_convs_2gpus")
graphdef, params,  state = nnx.split(model, nnx.Param,  ...)
params = checkpoint["params_ema"]

model = nnx.merge(graphdef, params, state)
sim_type = model.tokenizer.simulator
model.eval()


In [34]:
theta_mask_errors_long = pits_long_theta['theta_mask_errors']

In [35]:
mask_to_label(theta_masks[12], sim_type.model_types)

'Ball>WatsonStick'

In [36]:
pits_short_theta["theta_mask_support"]

array([[False, False, False, False, False, False, False, False, False,
        False,  True, False, False, False, False, False, False, False,
        False, False,  True],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True, False],
       [ True, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True, False],
       [ True, False, False, False, False, False, False, False, False,
        False,  True, False, False, False, False, False, False, False,
        False,  True, False],
       [False, False,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True, False],
       [False, False, False, False, False, False, False, False, False,
        False, False,  True, False, False, False, False, False, False

In [37]:
labels = theta_mask_labels
counts = theta_mask_counts
errors = theta_mask_errors
param_counts = counts
cmap_name = "viridis"  # "plasma"

In [38]:
pits_short_mask

NpzFile 'outputs/coverage_eval2/pit_coverage_mask_short.npz' with keys: alpha, coverage, pits, labels, hyperparameters...

In [39]:
counts

array([20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000,
       20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000,
       20000, 20000])

In [40]:
def random_error(rng, counts):
    u = jax.random.uniform(rng, shape=(counts,))
    ecdf = jnp.quantile(u, alpha)
    lower_bound = jnp.abs(ecdf - alpha).mean(-1)
    return lower_bound

In [41]:
random_error_thetas = [jax.vmap(random_error, in_axes=(0, None))(jax.random.split(jax.random.PRNGKey(42), 1000), counts) for counts in counts]

In [42]:
random_error_thetas_min = jnp.array([jnp.quantile(err, 0.05) for err in random_error_thetas]).min()
random_error_thetas_max  = jnp.array([jnp.quantile(err, 0.95) for err in random_error_thetas]).max()

In [43]:
random_error_mask = jax.vmap(random_error, in_axes=(0, None))(jax.random.split(jax.random.PRNGKey(42), 500), pits_s.shape[1])

In [44]:
random_error_mask_min = jnp.quantile(random_error_mask, 0.05)
random_error_mask_max = jnp.quantile(random_error_mask, 0.95)

In [45]:
color1 = '#1b7a52'
color2 = '#2ecf8b'

In [46]:
labels

array(['D', 'N', 'B', 'BD', 'S', 'S$_{\\mathcal{W}}$', 'SD', 'Z', 'BS',
       'DS$_{\\mathcal{W}}$', 'Z$_{\\mathcal{W}}$', 'ZD',
       'BS$_{\\mathcal{W}}$', 'BSD', 'DZ$_{\\mathcal{W}}$',
       'S$_{\\mathcal{B}}$', '2S', 'BZD', 'DNo$_{\\mathcal{W}}$', '2SD'],
      dtype='<U19')

In [47]:

fig, axes = plt.subplots(1, 2, figsize=(5., 2.2), width_ratios=[0.5, 1])
errors_short_mask = [jnp.abs(cov - alpha_s).mean() for cov in coverage_s]
errors_long_mask = [jnp.abs(cov - alpha_s).mean() for cov in coverage_l]

mask_short = ~np.isnan(hyperparameters_s)
axes[0].step(np.asarray(hyperparameters_s)[mask_short], np.asarray(errors_short_mask)[mask_short], color=color1, where='mid', label='n=HCP-like')
axes[0].step(np.asarray(hyperparameters_s)[mask_short], np.asarray(errors_long_mask)[mask_short], color=color2, where='mid', label='n=UKB-like')
axes[0].set_xlabel(r'Model complexity $\lambda$')
axes[0].set_ylabel('Calibration error')
axes[0].set_title('Model selection')
axes[0].fill_between(hyperparameters_s[mask_short],random_error_mask.min(), random_error_mask.max(), color="grey", alpha=0.15)
labels = labels[1:]
#labels = [label.replace('#', '') for label in labels]
errors = errors[1:]
errors_long = theta_mask_errors_long[1:]
positions = np.arange(len(labels))
axes[1].step(positions, errors, color=color1, where='mid', label='HCP-like')
axes[1].step(positions, errors_long, color=color2, where='mid', label='UKB-like')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False)
axes[1].set_xticklabels(labels, rotation=90, ha='right', fontsize=8)
axes[1].fill_between(positions, random_error_thetas_min, random_error_thetas_max, color="grey", alpha=0.15)
#axes[1].step(labels, random_error_thetas, color="grey", where='mid', label='Random', linestyle='--')
axes[1].set_xlabel('Model')
axes[1].set_xticks(positions)
axes[1].set_xticklabels(labels, rotation=90, ha='right', fontsize=8)
#axes[1].set_ylabel('Calibration error')
axes[1].set_title('Parameter inference')

#axes[1].grid(True, linestyle='--', alpha=0.3)
fig.tight_layout()
fig.savefig("coverage_plot.png", dpi=300)
fig.savefig("coverage_plot.svg")
plt.show()

/tmp/ipykernel_1463916/794062732.py:20: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(labels, rotation=90, ha='right', fontsize=8)


In [48]:
fig.show()

In [49]:
labels

array(['N', 'B', 'BD', 'S', 'S$_{\\mathcal{W}}$', 'SD', 'Z', 'BS',
       'DS$_{\\mathcal{W}}$', 'Z$_{\\mathcal{W}}$', 'ZD',
       'BS$_{\\mathcal{W}}$', 'BSD', 'DZ$_{\\mathcal{W}}$',
       'S$_{\\mathcal{B}}$', '2S', 'BZD', 'DNo$_{\\mathcal{W}}$', '2SD'],
      dtype='<U19')

In [54]:
overal_cov = jnp.quantile(pits_s.reshape(-1), alpha_s)

overal_cov_long = jnp.quantile(pits_l.reshape(-1), alpha_s)  # from long theta

In [55]:
overal_theta_cov = jnp.quantile(pits.reshape(-1), alpha_s)
overal_theta_cov_long = jnp.quantile(pits_theta_long.reshape(-1), alpha_s)

In [56]:
fig = plt.figure(figsize=(1.3,1.3))
plt.plot(alpha, alpha, linestyle='--', color='grey')
plt.plot(alpha, overal_cov, color=color1, lw =1)
plt.plot(alpha, overal_cov_long, color=color2, lw=1)
plt.xticks([0.0, 0.5, 1.0])
plt.yticks([0.0, 0.5, 1.0])
plt.xlabel('Nominal credible level')
plt.ylabel('Empirical credible level')
fig.suptitle('Overall model coverage')
fig.savefig("overall_model_coverage.svg", dpi=300)

In [57]:
fig = plt.figure(figsize=(1.3,1.3))
plt.plot(alpha, alpha, linestyle='--', color='grey')
plt.plot(alpha, overal_theta_cov, color=color1, lw=1)
plt.plot(alpha, overal_theta_cov_long, color=color2, lw=1)
plt.xticks([0.0, 0.5, 1.0])
plt.yticks([0.0, 0.5, 1.0])
plt.xlabel('Nominal credible level')
plt.ylabel('Empirical credible level')
fig.suptitle('Overall parameter coverage')
fig.savefig("overall_param_coverage.svg", dpi=300)

: 